# 🧬 GAJE HELIX — Crianza y Entrenamiento en GPU (Google Colab Pro)

Cuaderno oficial optimizado para entrenar organismos genómicos (`Q2_0`) con aceleración por **GPU (NVIDIA A100 / V100 / T4)** mediante shaders WGSL / Vulkan nativos en Rust con **anclaje en VRAM (Zero-Copy)**.

> ⚡ **Recomendación Colab Pro:** En `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución`, selecciona **GPU A100** o **T4** con memoria **High-RAM**.

### 🎮 Paso 1: Verificación de Hardware y Acelerador GPU

In [ ]:
# Inspeccionar GPU asignada por Colab Pro
!nvidia-smi

### 📦 Paso 2: Instalar Dependencias Vulkan para NVIDIA

In [ ]:
# Instalar librerías de enlace Vulkan para el driver de NVIDIA
!apt-get update -qq
!apt-get install -y -qq libvulkan1 libvulkan-dev vulkan-tools mesa-vulkan-drivers build-essential
!vulkaninfo --summary || true

### 🦀 Paso 3: Clonar Repositorio y Configurar Rust

In [ ]:
# Clonar la rama develop del repositorio oficial
!rm -rf /content/gaje-semantic-compression
!git clone -b develop https://github.com/erickaguilar/gaje-semantic-compression.git /content/gaje-semantic-compression
%cd /content/gaje-semantic-compression
!git submodule update --init --recursive

# Instalar compilador Rust moderno
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:" + os.environ['PATH']
!rustc --version
!cargo --version

### 🧬 Paso 4: Descargar Organismo Nacido `max_512_pro.gaje`

In [ ]:
# Descargar max_512_pro.gaje (208 MB) desde Hugging Face
!mkdir -p models/born
!curl -L -o models/born/max_512_pro.gaje https://huggingface.co/eaguilar/gaje-models/resolve/main/max_512_pro.gaje
!ls -lh models/born/max_512_pro.gaje

### 🚀 Paso 5: Compilar Núcleo Nativo GAJE con Soporte GPU

In [ ]:
# Compilación optimizada en release
!cargo build --release --bin gaje-cli

# Validar tests de GPU y memoria VRAM
!cargo test --test test_vram_viability -- --nocapture
!cargo test --test test_gpu_integration -- --nocapture

### 🔥 Paso 6: Crianza del Organismo con Aceleración GPU en Colab

In [ ]:
# Crianza con Ladder Training (-l 4) y aceleración GPU por 20 épocas
!./target/release/gaje-cli crianza \
    -m models/born/max_512_pro.gaje \
    -d data/genesis_conversational_corpus.jsonl \
    -e 20 \
    -l 4 \
    --gpu

### 💬 Paso 7: Prueba de Inferencia del Organismo Entrenado

In [ ]:
# Inferencia rápida
!./target/release/gaje-cli --model models/born/max_512_pro.gaje --prompt "¿Quién eres y qué puedes hacer?" --max-tokens 80

### 💾 Paso 8: Respaldo a Google Drive o Hugging Face

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/GAJE_Models
!cp models/born/max_512_pro.gaje /content/drive/MyDrive/GAJE_Models/max_512_pro_trained.gaje
!cp -r models/born/max_512_pro_memory /content/drive/MyDrive/GAJE_Models/ 2>/dev/null || true
print('✅ Organismo y memoria hipocampal respaldados en Google Drive.')